In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import os
import warnings
from scipy.stats import spearmanr
from scipy.stats import pearsonr
import matplotlib.patches as mpatches
from scipy import stats
%matplotlib inline
warnings.filterwarnings('ignore')

In [ ]:
# Define file paths for data and output figures
data_path = os.path.dirname(os.getcwd()) + '/data'
figure_path = os.path.dirname(os.getcwd()) + '/figures'

# Selected proteins to plot

In [ ]:
# Load unique proteins data
df = pd.read_excel(data_path + '/results/results.xlsx', 
                   sheet_name='Table S7 Unique Proteins', 
                   skiprows=1)

# Get set of all proteins that are unique to any subtype
all_unique_proteins = set(df['Protein'].tolist())
print(f"Total unique proteins across all subtypes: {len(all_unique_proteins)}")

# Get top 10 unique proteins for each immunophenotype sorted by Log2FC
aml_unique = df[df['Unique to:'] == 'AML'].sort_values('adj.P.Val_AML-Control', ascending=True).head(10)['Protein'].tolist()
bcp_unique = df[df['Unique to:'] == 'BCP-ALL'].sort_values('adj.P.Val_B-Control', ascending=True).head(10)['Protein'].tolist()
t_all_unique = df[df['Unique to:'] == 'T-ALL'].sort_values('adj.P.Val_T-Control', ascending=True).head(10)['Protein'].tolist()

# Display results
print(f"AML unique top {len(aml_unique)} proteins")
print(aml_unique)
print(f"\nBCP-ALL unique top {len(bcp_unique)} proteins")
print(bcp_unique)
print(f"\nT-ALL unique top {len(t_all_unique)} proteins")
print(t_all_unique)

In [ ]:
# Now load differential expression results for leukemia vs controls
df = pd.read_excel(data_path + '/results/results.xlsx', 
                   sheet_name='Table S3 Leukemia vs Control', 
                   skiprows=1)

In [ ]:
# Now load differential expression results for leukemia vs controls
df = pd.read_excel(data_path + '/results/results.xlsx', 
                   sheet_name='Table S3 Leukemia vs Control', 
                   skiprows=1)

# Filter for highly abundant proteins in leukemia
leukemia = df[(df['adj.P.Val'] < 0.05) & (df['Log2FC'] >= 1.5)]

# Remove proteins that are unique to any subtype
leukemia = leukemia[~leukemia['Protein'].isin(all_unique_proteins)]

# Sort by adj.P.Val
leukemia = leukemia.sort_values('adj.P.Val', ascending=True)

# Summary 
print(f"Leukemia: {len(leukemia)} highly abundant proteins (after removing subtype-unique proteins)")

# Get top 10 proteins sorted by adj.P.Val
leukemia_top_proteins = leukemia.head(10)['Protein'].tolist()

print(f"\nTop 10 most abundant proteins (excluding subtype-unique): {leukemia_top_proteins}")

In [ ]:
# Load BCP-ALL subtype specific biomarkers
df = pd.read_excel(data_path + '/results/results.xlsx', 
                   sheet_name='Table S15 HeH vs ER-overlap', 
                   skiprows=1)

# Create lists for each subtype
ER_top_proteins = df[df['Up regulated group'] == 'ETV6::RUNX1']['Protein /gene'].tolist()
HeH_top_proteins = df[df['Up regulated group'] == 'HeH']['Protein /gene'].tolist()

# Display results
print(f"ETV6::RUNX1 proteins: {len(ER_top_proteins)}")
print(ER_top_proteins)
print(f"\nHeH proteins: {len(HeH_top_proteins)}")
print(HeH_top_proteins)

# Olink data and clinical information

In [ ]:
# Clinical info
clin_info = pd.read_excel(data_path + '/results/results.xlsx', sheet_name='Table S1 - Sample QC', skiprows=1)

# Clinical variables to plot
clinical_vars = ['White blood cells', 'Blasts', 'Hemoglobin', 'Platelets']

# Remove Control patients from the dataframe
clin_filtered = clin_info[clin_info['Immunophenotype'] != 'Control'].copy()

# Rename B-ALL to BCP-ALL in the dataframe for consistency
clin_filtered['Immunophenotype'] = clin_filtered['Immunophenotype'].replace('B-ALL', 'BCP-ALL')

# Replace obvious missing data codes with NaN
clin_filtered['Blasts'] = clin_filtered['Blasts'].replace(999.0, np.nan)

In [ ]:
# Load and preprocess Olink proteomics data
olink = pd.read_csv(data_path + '/raw/olink.csv', delimiter=';')
olink = olink[['SampleID', 'Assay', 'NPX']]  # Keep only essential columns
print('Olink data - number of unique Sample IDs', len(set(olink['SampleID'])))

# Pivot data from long to wide format (samples as rows, proteins as columns)
pivoted_data = olink.pivot_table(index='SampleID', columns='Assay', values='NPX', aggfunc='mean')
pivoted_data.columns.name = None
pivoted_data = pivoted_data.reset_index()

# Load sample phenotype information
pheno = pd.read_excel(data_path + '/raw/MASTER_PHENO_FILE_AML_ALL_controls_20250607.xlsx')

# Clean immunopheno column - replace missing values with 'CTRL' (control)
pheno['immunopheno'] = pheno['immunopheno'].fillna('CTRL').replace(['nan', 'NaN', None], 'CTRL')
pheno['SampleID'] = pheno['sample_id']
print('Pheno data - number of unique Sample IDs', len(set(pheno['SampleID'])))
pheno = pheno[['SampleID', 'public_id', 'immunopheno', 'Subtype']]

# Merge proteomics data with phenotype information
merged = pd.merge(pivoted_data, pheno, on='SampleID', how='inner')

# Remove specific patients (quality control exclusions)
sample_ALL_920 = merged[merged['public_id'] == 'ALL_920']
removed_patients = ['AML_101','AML_139', 'ALL_920', 'K-023']
final_df = merged[~merged['public_id'].isin(removed_patients)]
print('Merged data - number of unique Sample IDs', len(set(final_df['SampleID'])))

print('----------')
print('Sample distribution across immunophenotypes')
counts = final_df['immunopheno'].value_counts()
print(counts)

In [ ]:
# Drop SampleID and rename public_id to SampleID in one step
final_df = final_df.drop('SampleID', axis=1).rename(columns={'public_id': 'SampleID'})

clin_variables = clin_filtered.copy()
clin_variables['SampleID'] = clin_variables['Public Sample ID']
clin_variables = clin_variables[['SampleID', 'White blood cells', 'Blasts', 'Platelets', 'Hemoglobin']]

# Merge proteomics data with phenotype information
clin_merged = pd.merge(final_df, clin_variables, on='SampleID', how='left')
clin_merged['immunopheno'] = clin_merged['immunopheno'].replace('B-ALL', 'BCP-ALL')
clin_merged['Subtype'] = clin_merged['Subtype'].replace('t1221', 'ETV6::RUNX1')

# Main plots

In [ ]:
# Combine all protein lists into one (leukemia + 3 immunophenotype-specific)
all_proteins = leukemia_top_proteins + aml_unique + bcp_unique + t_all_unique

print(f"Total proteins to plot: {len(all_proteins)}")
print(f"  Leukemia general: {len(leukemia_top_proteins)}")
print(f"  AML unique: {len(aml_unique)}")
print(f"  BCP-ALL unique: {len(bcp_unique)}")
print(f"  T-ALL unique: {len(t_all_unique)}")

# FIRST: Calculate control averages from the ORIGINAL data before any filtering
print("\nCalculating control averages from original data...")
control_averages = {}
mask_ctrl_original = clin_merged['immunopheno'] == 'CTRL'

for protein in all_proteins:
    if protein in clin_merged.columns:
        control_avg = clin_merged.loc[mask_ctrl_original, protein].mean()
        n_ctrl = clin_merged.loc[mask_ctrl_original, protein].notna().sum()
        control_averages[protein] = control_avg
        print(f"  {protein}: {control_avg:.2f} (n={n_ctrl})")
    else:
        print(f"  {protein}: NOT FOUND IN DATA")

# Add control averages for subtype proteins too
for protein in ER_top_proteins + HeH_top_proteins:
    if protein not in control_averages and protein in clin_merged.columns:
        control_avg = clin_merged.loc[mask_ctrl_original, protein].mean()
        control_averages[protein] = control_avg

# NOW prepare the data - REMOVE non-control samples with missing blasts, and REMOVE controls entirely
df_plot = clin_merged.copy()

# Track which samples originally had NaN blasts
df_plot['Blasts_was_missing'] = df_plot['Blasts'].isna()

# Keep only patients with valid blast data (exclude controls)
mask_keep = (df_plot['immunopheno'] != 'CTRL') & (~df_plot['Blasts_was_missing'])
df_plot = df_plot[mask_keep].copy()

print(f"\nSamples after filtering: {len(df_plot)}")
print(f"Patient samples with valid blast data: {len(df_plot)}")

# Define colors for immunophenotypes
color_map = {
    'AML': "#EFB2D1",      # Yellow/gold
    'BCP-ALL': "#447597",  # Orange
    'T-ALL': '#87CCB2'     # Red
}

# Get unique immunophenotypes
immunopheno_categories = df_plot['immunopheno'].dropna().unique()

# Create protein groups with labels
protein_groups = [
    ("Leukemia (General)", leukemia_top_proteins),
    ("AML", aml_unique),
    ("BCP-ALL", bcp_unique),
    ("T-ALL", t_all_unique)
]

# Function to create a plot for a protein group
def plot_protein_group(proteins, group_name, filename_base):
    n_proteins = len(proteins)
    n_cols = 5
    n_rows = int(np.ceil(n_proteins / n_cols))
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 3.5*n_rows))
    if n_rows == 1:
        axes = axes.reshape(1, -1)
    axes = axes.flatten()
    
    # Add main title
    fig.suptitle(group_name, fontsize=16, fontweight='bold', y=0.995)
    
    # Create scatter plots for each protein
    for idx, protein in enumerate(proteins):
        ax = axes[idx]
        
        # Check if protein exists in dataframe
        if protein not in df_plot.columns:
            ax.text(0.5, 0.5, f'{protein}\nNot found in data', 
                    ha='center', va='center', transform=ax.transAxes, fontsize=10)
            ax.set_title(f'{protein} (Not Available)', fontsize=10)
            continue
        
        # Plot each immunophenotype
        for immuno in immunopheno_categories:
            mask = df_plot['immunopheno'] == immuno
            
            if mask.sum() > 0:
                ax.scatter(df_plot.loc[mask, 'Blasts'], 
                          df_plot.loc[mask, protein],
                          label=immuno, 
                          #alpha=0.6, 
                          s=60,
                          color=color_map.get(immuno, "#474646"),
                          edgecolors='grey',
                          linewidths=1)
        
        # Add horizontal dotted line for control average
        if protein in control_averages and not np.isnan(control_averages[protein]):
            ax.axhline(y=control_averages[protein], color='black', linestyle='--', 
                       linewidth=1.5, alpha=0.5, zorder=1)
            ax.text(0.02, control_averages[protein], f' {control_averages[protein]:.2f}', 
                    verticalalignment='bottom', fontsize=7, color='black',
                    transform=ax.get_yaxis_transform())
        
        ax.set_xlabel('Blast %', fontsize=9)
        ax.set_ylabel('NPX', fontsize=9)
        ax.set_title(f'{protein}', fontsize=10, fontweight='bold')
        ax.grid(True, alpha=0.3)
        ax.tick_params(labelsize=8)
    
    # Remove empty subplots
    for idx in range(n_proteins, len(axes)):
        fig.delaxes(axes[idx])
    
    # Create legend separately at the bottom
    from matplotlib.patches import Patch
    from matplotlib.lines import Line2D
    

    # Desired legend order
    desired_order = ['AML', 'T-ALL', 'BCP-ALL']

    legend_elements = []

    # 1️⃣ Controls first
    legend_elements.append(
        Line2D([0], [0], color='black', linestyle='--', linewidth=1.5, label='Controls (average)'))

    # 2️⃣ Then disease groups in desired order
    for immuno in desired_order:
        if immuno in immunopheno_categories:
            legend_elements.append(
                Patch(facecolor=color_map[immuno],
                    label=immuno)
            )

    
    fig.legend(handles=legend_elements, loc='lower center', ncol=4, 
              fontsize=11, frameon=True, fancybox=True, shadow=True,
              bbox_to_anchor=(0.5, -0.04))
    
    plt.tight_layout()
    
    # Save the figure
    png_path = os.path.join(figure_path, f'{filename_base}.png')
    pdf_path = os.path.join(figure_path, f'{filename_base}.pdf')
    
    fig.savefig(png_path, dpi=300, bbox_inches='tight')
    fig.savefig(pdf_path, bbox_inches='tight')
    
    print(f"Saved: {filename_base}.png and {filename_base}.pdf")
    
    plt.show()

# Create separate plots for each group
group_filenames = [
    "blast_correlation_leukemia_general",
    "blast_correlation_AML",
    "blast_correlation_BCP-ALL",
    "blast_correlation_T-ALL"
]

for (group_name, proteins), filename in zip(protein_groups, group_filenames):
    if len(proteins) > 0:
        plot_protein_group(proteins, group_name, filename)

# ============================================================================
# BCP-ALL SUBTYPE DIFFERENTIATION
# ============================================================================

# Check if Subtype column exists
if 'Subtype' not in df_plot.columns:
    print("\nWarning: 'Subtype' column not found in data!")
else:
    # Filter for BCP-ALL samples only
    df_bcp = df_plot[df_plot['immunopheno'] == 'BCP-ALL'].copy()
    
    print(f"\nBCP-ALL samples: {len(df_bcp)}")
    print(f"Subtypes available:")
    print(df_bcp['Subtype'].value_counts())
    
    # Define colors for subtypes
    subtype_color_map = {
        'HeH': '#FF8C42',
        'ETV6::RUNX1': '#6A4C93'
    }
    
    # Get unique subtypes
    subtype_categories = df_bcp['Subtype'].dropna().unique()
    
    # Function to plot BCP-ALL subtype proteins
    def plot_bcp_subtype_group(proteins, group_title, filename_base):
        n_proteins = len(proteins)
        n_cols = 5
        n_rows = int(np.ceil(n_proteins / n_cols))
        
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 3.5*n_rows))
        if n_rows == 1 and n_cols == 1:
            axes = np.array([axes])
        elif n_rows == 1:
            axes = axes.reshape(1, -1)
        axes = axes.flatten()
        
        # Add main title
        fig.suptitle(group_title, fontsize=16, fontweight='bold', y=0.995)
        
        for idx, protein in enumerate(proteins):
            ax = axes[idx]
            
            if protein not in df_bcp.columns:
                ax.text(0.5, 0.5, f'{protein}\nNot found in data', 
                        ha='center', va='center', transform=ax.transAxes, fontsize=10)
                ax.set_title(f'{protein} (Not Available)', fontsize=10)
                continue
            
            # Plot each subtype with different color
            for subtype in subtype_categories:
                mask = df_bcp['Subtype'] == subtype
                
                if mask.sum() > 0:
                    # Determine label
                    if subtype == 'ETV6::RUNX1':
                        label = r'$\mathit{ETV6::RUNX1}$'
                    else:
                        label = subtype
                    
                    ax.scatter(df_bcp.loc[mask, 'Blasts'], 
                              df_bcp.loc[mask, protein],
                              label=label, 
                              alpha=0.9, 
                              s=60,
                              color=subtype_color_map.get(subtype, "#474646"),
                              edgecolors='grey',
                              linewidths=1)
            
            # Add horizontal dotted line for control average
            if protein in control_averages and not np.isnan(control_averages[protein]):
                ax.axhline(y=control_averages[protein], color='black', linestyle='--', 
                           linewidth=1.5, alpha=0.5, zorder=1)
                ax.text(0.02, control_averages[protein], f' {control_averages[protein]:.2f}', 
                        verticalalignment='bottom', fontsize=7, color='black',
                        transform=ax.get_yaxis_transform())
            
            ax.set_xlabel('Blast %', fontsize=9)
            ax.set_ylabel('NPX', fontsize=9)
            ax.set_title(f'{protein}', fontsize=10, fontweight='bold')
            ax.grid(True, alpha=0.3)
            ax.tick_params(labelsize=8)
        
        # Remove empty subplots
        for idx in range(n_proteins, len(axes)):
            fig.delaxes(axes[idx])
        
        # Create legend at the bottom
        from matplotlib.patches import Patch
        from matplotlib.lines import Line2D
        
        legend_elements = []
        for subtype in subtype_categories:
            if subtype == 'ETV6::RUNX1':
                label = r'$\mathit{ETV6::RUNX1}$'
            else:
                label = subtype
            legend_elements.append(Patch(facecolor=subtype_color_map.get(subtype, "#434141"), 
                                        alpha=0.7, label=label))
        legend_elements.append(Line2D([0], [0], color='black', linestyle='--', 
                                      linewidth=1.5, label='Controls (average)'))
        
        fig.legend(handles=legend_elements, loc='lower center', ncol=3, 
                  fontsize=11, frameon=True, fancybox=True, shadow=True,
                  bbox_to_anchor=(0.5, -0.04))
        
        plt.tight_layout()
        
        # Save the figure
        png_path = os.path.join(figure_path, f'{filename_base}.png')
        pdf_path = os.path.join(figure_path, f'{filename_base}.pdf')
        
        fig.savefig(png_path, dpi=300, bbox_inches='tight')
        fig.savefig(pdf_path, bbox_inches='tight')
        
        print(f"Saved: {filename_base}.png and {filename_base}.pdf")
        
        plt.show()
    
    # Plot ETV6::RUNX1 proteins
    if len(ER_top_proteins) > 0:
        plot_bcp_subtype_group(ER_top_proteins, 
                               'BCP-ALL Subtype: ETV6::RUNX1 Upregulated Proteins',
                               'blast_correlation_BCP-ALL_subtype_ETV6-RUNX1')
    
    # Plot HeH proteins
    if len(HeH_top_proteins) > 0:
        plot_bcp_subtype_group(HeH_top_proteins, 
                               'BCP-ALL Subtype: HeH Upregulated Proteins',
                               'blast_correlation_BCP-ALL_subtype_HeH')
    
    # Print statistics by subtype
    print("\n" + "-"*60)
    print("STATISTICS BY BCP-ALL SUBTYPE")
    print("-"*60)
    
    for protein_list, list_name in [(ER_top_proteins, 'ETV6::RUNX1'), (HeH_top_proteins, 'HeH')]:
        print(f"\n{list_name} PROTEINS:")
        print("=" * 60)
        
        for protein in protein_list:
            if protein not in df_bcp.columns:
                continue
            
            print(f"\n{protein}:")
            print("-" * 50)
            
            for subtype in subtype_categories:
                mask = df_bcp['Subtype'] == subtype
                values = df_bcp.loc[mask, protein].dropna()
                
                if len(values) > 0:
                    mean_val = values.mean()
                    std_val = values.std()
                    median_val = values.median()
                    n = len(values)
                    print(f"  {subtype:20s}: n={n:3d}, Mean={mean_val:6.2f}, Std={std_val:6.2f}, Median={median_val:6.2f}")

# Extra sample from the same patient

In [ ]:
protein_list_er_heh = ['DSC2','PTPRK', 'IL6R','ADAM8']
sample_ALL_920 = sample_ALL_920[protein_list_er_heh]
sample_ALL_920

In [ ]:
sample_ALL_1428 = df_bcp[df_bcp['SampleID'] == 'ALL_1428'] 
sample_ALL_1428 = sample_ALL_1428[protein_list_er_heh]
sample_ALL_1428

In [ ]:
# Key Proteins from Each Immunophenotype

# Define the 4 key proteins
key_proteins = ['TK1', 'CEBPA', 'NOTCH1', 'SIGLEC15']
protein_labels = ['TK1\n(Leukemia General)', 'CEBPA\n(AML)', 'NOTCH1\n(T-ALL)', 'SIGLEC15\n(BCP-ALL)',]

# Create figure with 4 subplots in one row
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

# Define colors for immunophenotypes
color_map = {
    'AML': "#EFB2D1",      # Yellow/gold
    'BCP-ALL': "#447597",  # Orange
    'T-ALL': '#87CCB2'     # Red
}

immunopheno_categories = df_plot['immunopheno'].dropna().unique()

# Create scatter plots for each protein
for idx, (protein, label) in enumerate(zip(key_proteins, protein_labels)):
    ax = axes[idx]
    
    # Check if protein exists in dataframe
    if protein not in df_plot.columns:
        ax.text(0.5, 0.5, f'{protein}\nNot found in data', 
                ha='center', va='center', transform=ax.transAxes, fontsize=10)
        ax.set_title(label, fontsize=11, fontweight='bold')
        continue
    
    # Plot each immunophenotype
    for immuno in immunopheno_categories:
        mask = df_plot['immunopheno'] == immuno
        
        if mask.sum() > 0:
            ax.scatter(df_plot.loc[mask, 'Blasts'], 
                      df_plot.loc[mask, protein],
                      label=immuno, 
                      alpha=0.9, 
                      s=60,
                      color=color_map.get(immuno, "#474646"),
                      edgecolors='grey',
                      linewidths=0.5)
    
    # Add horizontal dotted line for control average
    if protein in control_averages and not np.isnan(control_averages[protein]):
        ax.axhline(y=control_averages[protein], color='black', linestyle='--', 
                   linewidth=1.5, alpha=0.7, zorder=1)
        ax.text(0.02, control_averages[protein], f' {control_averages[protein]:.2f}', 
                verticalalignment='bottom', fontsize=12, color='black',
                transform=ax.get_yaxis_transform())
    
    ax.set_xlabel('Blast %', fontsize=11)
    ax.set_ylabel('NPX', fontsize=11)
    ax.set_title(label, fontsize=12)
    ax.grid(True, alpha=0.3)
    ax.tick_params(labelsize=11)

# Create legend at the bottom
from matplotlib.patches import Patch
from matplotlib.lines import Line2D

# Desired legend order
desired_order = ['AML', 'T-ALL', 'BCP-ALL']

legend_elements = []

# 1️⃣ Controls first
legend_elements.append(Line2D([0], [0], color='black', linestyle='--', linewidth=1.5, label='Controls (average)'))

# 2️⃣ Then disease groups in desired order
for immuno in desired_order:
    if immuno in immunopheno_categories:
        legend_elements.append(
            Patch(facecolor=color_map[immuno], label=immuno))

fig.legend(handles=legend_elements, loc='lower center', ncol=4, 
          fontsize=11, frameon=True, fancybox=True, shadow=True,
          bbox_to_anchor=(0.5, -0.08))

plt.tight_layout()

# Save the figure
png_path = os.path.join(figure_path, 'blast_correlation_key_proteins.png')
pdf_path = os.path.join(figure_path, 'blast_correlation_key_proteins.pdf')
pdf_path = os.path.join(figure_path, 'blast_correlation_key_proteins.svg')

fig.savefig(png_path, dpi=300, bbox_inches='tight')
fig.savefig(pdf_path, bbox_inches='tight')

plt.show()


# BCP-ALL Subtype Key Proteins

# Check if we have BCP-ALL data
if 'Subtype' in df_plot.columns:
    # Filter for BCP-ALL samples only
    df_bcp = df_plot[df_plot['immunopheno'] == 'BCP-ALL'].copy()
    
    # Define the 4 key proteins for subtypes
    subtype_proteins = ['DSC2', 'PTPRK', 'IL6R', 'ADAM8']
    subtype_labels = ['DSC2\n($\mathit{ETV6::RUNX1}$)', 'PTPRK\n($\mathit{ETV6::RUNX1}$)', 'IL6R\n(HeH)', 'ADAM8\n(HeH)']
    
    # Create figure with 4 subplots in one row
    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    
    # Define colors for subtypes
    subtype_color_map = {
        'HeH': '#FF8C42',
        'ETV6::RUNX1': '#6A4C93'
    }
    
    subtype_categories = df_bcp['Subtype'].dropna().unique()
    
    # Create scatter plots for each protein
    for idx, (protein, label) in enumerate(zip(subtype_proteins, subtype_labels)):
        ax = axes[idx]
        
        if protein not in df_bcp.columns:
            ax.text(0.5, 0.5, f'{protein}\nNot found in data', 
                    ha='center', va='center', transform=ax.transAxes, fontsize=10)
            ax.set_title(label, fontsize=11, fontweight='bold')
            continue
        
        # Plot each subtype with different color
        for subtype in subtype_categories:
            mask = df_bcp['Subtype'] == subtype
            
            if mask.sum() > 0:
                # Determine label for legend
                if subtype == 'ETV6::RUNX1':
                    legend_label = r'$\mathit{ETV6::RUNX1}$'
                else:
                    legend_label = subtype
                
                ax.scatter(df_bcp.loc[mask, 'Blasts'], 
                          df_bcp.loc[mask, protein],
                          label=legend_label, 
                          alpha=0.9, 
                          s=60,
                          color=subtype_color_map.get(subtype, "#474646"),
                          edgecolors='grey',
                          linewidths=0.5)
        
        # Add horizontal dotted line for control average
        if protein in control_averages and not np.isnan(control_averages[protein]):
            ax.axhline(y=control_averages[protein], color='black', linestyle='--', 
                       linewidth=1.5, alpha=0.7, zorder=1)
            ax.text(0.02, control_averages[protein], f' {control_averages[protein]:.2f}', 
                    verticalalignment='bottom', fontsize=12, color='black',
                    transform=ax.get_yaxis_transform())
        
        ax.set_xlabel('Blast %', fontsize=11)
        ax.set_ylabel('NPX', fontsize=11)
        ax.set_ylim(-1, 5.5) 
        ax.set_title(label, fontsize=12)
        ax.grid(True, alpha=0.3)
        ax.tick_params(labelsize=11)
    
    # Create legend at the bottom
    legend_elements = []
    for subtype in subtype_categories:
        if subtype == 'ETV6::RUNX1':
            legend_label = r'$\mathit{ETV6::RUNX1}$'
        else:
            legend_label = subtype
        legend_elements.append(Patch(facecolor=subtype_color_map.get(subtype, "#434141"), 
                                    alpha=0.9, label=legend_label))
    legend_elements.append(Line2D([0], [0], color='black', linestyle='--', 
                                  linewidth=2, label='Controls (average)'))
    
    fig.legend(handles=legend_elements, loc='lower center', ncol=3, 
              fontsize=12, frameon=True, fancybox=True, shadow=True,
              bbox_to_anchor=(0.5, -0.08))
    
    plt.tight_layout()
    
    # Save the figure
    png_path = os.path.join(figure_path, 'blast_correlation_BCP-ALL_subtype_key_proteins.png')
    pdf_path = os.path.join(figure_path, 'blast_correlation_BCP-ALL_subtype_key_proteins.pdf')
    pdf_path = os.path.join(figure_path, 'blast_correlation_BCP-ALL_subtype_key_proteins.svg')
    
    fig.savefig(png_path, dpi=300, bbox_inches='tight')
    fig.savefig(pdf_path, bbox_inches='tight')
    
    plt.show()
else:
    print("\nWarning: 'Subtype' column not found - skipping BCP-ALL subtype plot")
